In [0]:
from pyspark.sql import functions as F

# Read Silver tables
shrinkage = spark.table(
    "workforce_analytics.silver.fact_shrinkage"
).alias("f")

people = spark.table(
    "workforce_analytics.silver.dim_people"
).alias("p")

shrinkage_type = spark.table(
    "workforce_analytics.silver.dim_shrinkage_type"
).alias("s")

# Join dimensions
fact_shrinkage_gold = (
    shrinkage
    .join(
        people,
        F.col("f.people_id") == F.col("p.people_id"),
        "left"
    )
    .join(
        shrinkage_type,
        F.col("f.shrinkage_type_id") ==
        F.col("s.shrinkage_type_id"),
        "left"
    )
    .select(
        *[F.col(f"f.{col}") for col in shrinkage.columns],

        # Employee attributes
        F.col("p.employee_name"),
        F.col("p.job_role"),
        F.col("p.employment_type"),
        F.col("p.team"),

        # Shrinkage type attributes
        F.col("s.shrinkage_type"),
        F.col("s.category"),
        F.col("s.is_paid"),
        F.col("s.is_planned")
    )
)

# Save Gold table
(
    fact_shrinkage_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workforce_analytics.gold.fact_shrinkage_gold"
    )
)

print("Gold Fact Shrinkage created successfully.")

In [0]:
gold_shrinkage = spark.table(
    "workforce_analytics.gold.fact_shrinkage_gold"
)

print("Snapshot counts:")

display(
    gold_shrinkage.groupBy("snapshot_date")
    .count()
    .orderBy("snapshot_date")
)

print("Total rows:", gold_shrinkage.count())

print("Duplicate shrinkage IDs:")

display(
    gold_shrinkage.groupBy("snapshot_date", "shrinkage_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Missing employee names:",
    gold_shrinkage.filter(
        F.col("employee_name").isNull()
    ).count()
)

print(
    "Missing shrinkage types:",
    gold_shrinkage.filter(
        F.col("shrinkage_type").isNull()
    ).count()
)